# Dysarthric Speech Detection via Late Fusion of Temporal MFCC Sequences and Prosodic Features using 1D-CNN

**Research Contribution:**  
This notebook implements a novel late-fusion architecture that addresses two key gaps in existing TORGO literature:
1. Prior work discards temporal dynamics by frame-averaging MFCCs before classification
2. No prior TORGO binary detection paper fuses temporal MFCC sequences with hand-crafted prosodic features at the representation level

**Ablation Study:** Three conditions are evaluated:
- **Baseline**: SVM on frame-averaged MFCCs (replicates prior work)
- **Model B**: 1D-CNN on temporal MFCC sequences only  
- **Model C (Proposed)**: 1D-CNN + Prosodic late fusion

**Dataset:** TORGO Database (Rudzicz et al., 2012)

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm import tqdm

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    accuracy_score
)
from sklearn.pipeline import Pipeline

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, ReLU,
    GlobalAveragePooling1D, Dense, Dropout,
    Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Prosodic feature extraction
try:
    import parselmouth
    from parselmouth.praat import call
    PARSELMOUTH_AVAILABLE = True
except ImportError:
    print("parselmouth not found. Install with: pip install praat-parselmouth")
    PARSELMOUTH_AVAILABLE = False

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 13
sns.set_style('darkgrid')

print("TensorFlow version:", tf.__version__)
print("Setup complete.")

## 2. Data Loading and EDA

In [ ]:
# ── Update this path to your TORGO dataset location ──
DATASET_DIR = '../input/dysarthria-detection'
CSV_PATH    = os.path.join(DATASET_DIR, 'torgo_data/data.csv')

data = pd.read_csv(CSV_PATH)
data['filepath'] = data['filename'].apply(
    lambda x: os.path.join(DATASET_DIR, x)
)

print(f"Total samples: {len(data)}")
print("\nClass distribution:")
print(data['is_dysarthria'].value_counts())
print("\nGender distribution:")
print(data['gender'].value_counts())
data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class balance
data['is_dysarthria'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'coral'],
    edgecolor='black'
)
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(rotation=0)

# Gender split per class
pd.crosstab(data['is_dysarthria'], data['gender']).plot(
    kind='bar', ax=axes[1], color=['salmon', 'cornflowerblue'],
    edgecolor='black'
)
axes[1].set_title('Gender Distribution per Class')
axes[1].set_xlabel('Class')
axes[1].tick_params(rotation=0)

plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualise one dysarthric vs one healthy waveform + spectrogram
sample_dys = data[data['is_dysarthria'] == 'dysarthria']['filepath'].iloc[0]
sample_non = data[data['is_dysarthria'] == 'non_dysarthria']['filepath'].iloc[0]

y_dys, sr = librosa.load(sample_dys, sr=16000)
y_non, sr = librosa.load(sample_non, sr=16000)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Waveforms
librosa.display.waveshow(y_non, sr=sr, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Healthy Speech — Waveform')

librosa.display.waveshow(y_dys, sr=sr, ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Dysarthric Speech — Waveform')

# Mel spectrograms
for ax, y, title, cmap in zip(
    [axes[1, 0], axes[1, 1]],
    [y_non, y_dys],
    ['Healthy Speech — Mel Spectrogram', 'Dysarthric Speech — Mel Spectrogram'],
    ['Blues', 'Reds']
):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_db, sr=sr, x_axis='time',
                                    y_axis='mel', ax=ax, cmap=cmap)
    ax.set_title(title)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.savefig('waveform_spectrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# MFCC comparison: healthy vs dysarthric
mfcc_non = librosa.feature.mfcc(y=y_non, sr=sr, n_mfcc=40)
mfcc_dys = librosa.feature.mfcc(y=y_dys, sr=sr, n_mfcc=40)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, mfcc, title, cmap in zip(
    axes,
    [mfcc_non, mfcc_dys],
    ['Healthy Speech — 40-dim MFCC (Temporal)', 'Dysarthric Speech — 40-dim MFCC (Temporal)'],
    ['Blues', 'Reds']
):
    img = librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=ax, cmap=cmap)
    ax.set_title(title)
    ax.set_ylabel('MFCC Coefficient')
    fig.colorbar(img, ax=ax)

plt.tight_layout()
plt.savefig('mfcc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Note: Temporal MFCC shape preserved — NOT frame-averaged.")
print(f"MFCC shape (n_mfcc x time_frames): {mfcc_non.shape}")

## 3. Feature Extraction

### 3.1 Temporal MFCC Sequences (40-dim, T=128 frames)
Unlike prior work that averages MFCC frames into a single vector, we preserve the full temporal sequence. This captures the irregular articulation dynamics characteristic of dysarthric speech.

### 3.2 Prosodic Features (7-dim)
We extract clinically-motivated hand-crafted prosodic features: jitter, shimmer, HNR, mean F0, std F0, speaking rate, and voice break percentage.

In [ ]:
# ── Configuration ──────────────────────────────────────────
N_MFCC     = 40    # MFCC dimensions (standard in literature)
MAX_FRAMES = 128   # Fixed temporal length (pad/truncate)
SR         = 16000 # Sampling rate (TORGO standard)
N_PROSODIC = 7     # Number of prosodic features
# ───────────────────────────────────────────────────────────

def extract_mfcc_sequence(filepath, n_mfcc=N_MFCC, max_frames=MAX_FRAMES, sr=SR):
    """
    Extract temporal MFCC sequence, preserving time dimension.
    Returns shape: (max_frames, n_mfcc) — input for 1D-CNN.
    """
    try:
        y, _ = librosa.load(filepath, sr=sr)
        # Pre-emphasis
        y = librosa.effects.preemphasis(y)
        # Extract MFCCs with delta and delta-delta
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                                     hop_length=160, n_fft=512)
        # Transpose: (n_mfcc, T) → (T, n_mfcc)
        mfcc = mfcc.T
        # Pad or truncate to fixed length
        if mfcc.shape[0] < max_frames:
            pad = max_frames - mfcc.shape[0]
            mfcc = np.pad(mfcc, ((0, pad), (0, 0)), mode='constant')
        else:
            mfcc = mfcc[:max_frames, :]
        return mfcc.astype(np.float32)
    except Exception as e:
        return np.zeros((max_frames, n_mfcc), dtype=np.float32)


def extract_prosodic_features(filepath, sr=SR):
    """
    Extract 7 clinically-motivated prosodic features using Praat (parselmouth).
    Features: jitter, shimmer, HNR, F0 mean, F0 std, speaking rate, voice break %
    Returns shape: (7,)
    """
    features = np.zeros(N_PROSODIC, dtype=np.float32)

    if not PARSELMOUTH_AVAILABLE:
        # Fallback: librosa-based approximate prosodic features
        try:
            y, _ = librosa.load(filepath, sr=sr)
            f0, voiced_flag, _ = librosa.pyin(
                y, fmin=75, fmax=500, sr=sr
            )
            f0_valid = f0[voiced_flag > 0]
            features[0] = 0.0   # jitter placeholder
            features[1] = 0.0   # shimmer placeholder
            features[2] = 0.0   # HNR placeholder
            features[3] = float(np.nanmean(f0_valid)) if len(f0_valid) > 0 else 0.0
            features[4] = float(np.nanstd(f0_valid))  if len(f0_valid) > 0 else 0.0
            # Approximate speaking rate via zero crossing rate
            zcr = librosa.feature.zero_crossing_rate(y)
            features[5] = float(np.mean(zcr))
            features[6] = float(1.0 - np.mean(voiced_flag)) if voiced_flag is not None else 0.0
        except Exception:
            pass
        return np.nan_to_num(features)

    try:
        sound = parselmouth.Sound(filepath)

        # Jitter (local pitch perturbation)
        pp = call(sound, "To PointProcess (periodic, cc)", 75, 500)
        jitter = call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        features[0] = float(jitter) if jitter and not np.isnan(jitter) else 0.0

        # Shimmer (local amplitude perturbation)
        shimmer = call([sound, pp], "Get shimmer (local)",
                       0, 0, 0.0001, 0.02, 1.3, 1.6)
        features[1] = float(shimmer) if shimmer and not np.isnan(shimmer) else 0.0

        # Harmonics-to-Noise Ratio
        harmonicity = call(sound, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, "Get mean", 0, 0)
        features[2] = float(hnr) if hnr and not np.isnan(hnr) else 0.0

        # F0 (Fundamental Frequency) — mean and std
        pitch = call(sound, "To Pitch", 0, 75, 500)
        f0_mean = call(pitch, "Get mean", 0, 0, "Hertz")
        f0_std  = call(pitch, "Get standard deviation", 0, 0, "Hertz")
        features[3] = float(f0_mean) if f0_mean and not np.isnan(f0_mean) else 0.0
        features[4] = float(f0_std)  if f0_std  and not np.isnan(f0_std)  else 0.0

        # Approximate speaking rate (syllables/sec via voiced fraction)
        total_frames = call(pitch, "Get number of frames")
        voiced_frames = call(pitch, "Count voiced frames")
        duration = call(sound, "Get total duration")
        voiced_frac = voiced_frames / total_frames if total_frames > 0 else 0.0
        features[5] = voiced_frac / duration if duration > 0 else 0.0

        # Voice break percentage
        vb_pct = call(pp, "Get fraction of locally unvoiced frames",
                      0, 0, 0.0001, 0.02, 1.3, 1.6)
        features[6] = float(vb_pct) if vb_pct and not np.isnan(vb_pct) else 0.0

    except Exception:
        pass

    return np.nan_to_num(features)


def extract_averaged_mfcc(filepath, n_mfcc=N_MFCC, sr=SR):
    """
    Baseline: frame-averaged MFCC vector (replicates prior work).
    Returns shape: (n_mfcc,)
    """
    try:
        y, _ = librosa.load(filepath, sr=sr)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        return np.mean(mfcc, axis=1).astype(np.float32)
    except Exception:
        return np.zeros(n_mfcc, dtype=np.float32)


print("Feature extraction functions defined.")
print(f"MFCC sequence shape per sample: ({MAX_FRAMES}, {N_MFCC})")
print(f"Prosodic feature vector size: {N_PROSODIC}")
print(f"Baseline averaged MFCC size: {N_MFCC}")

In [ ]:
# ── Run feature extraction over entire dataset ──
# This may take several minutes depending on dataset size

filepaths = data['filepath'].tolist()
labels_raw = data['is_dysarthria'].tolist()

print(f"Extracting features for {len(filepaths)} audio files...")

X_mfcc_seq  = []   # Temporal MFCC sequences — for 1D-CNN
X_prosodic  = []   # Prosodic feature vectors — for fusion branch
X_mfcc_avg  = []   # Averaged MFCCs — for SVM baseline
y_labels    = []

for fp, lab in tqdm(zip(filepaths, labels_raw), total=len(filepaths)):
    if not os.path.exists(fp):
        continue
    X_mfcc_seq.append(extract_mfcc_sequence(fp))
    X_prosodic.append(extract_prosodic_features(fp))
    X_mfcc_avg.append(extract_averaged_mfcc(fp))
    y_labels.append(1 if lab == 'dysarthria' else 0)

X_mfcc_seq = np.array(X_mfcc_seq, dtype=np.float32)  # (N, 128, 40)
X_prosodic  = np.array(X_prosodic, dtype=np.float32)  # (N, 7)
X_mfcc_avg  = np.array(X_mfcc_avg, dtype=np.float32)  # (N, 40)
y           = np.array(y_labels, dtype=np.int32)        # (N,)

print(f"\nExtraction complete.")
print(f"MFCC sequence array: {X_mfcc_seq.shape}")
print(f"Prosodic array:      {X_prosodic.shape}")
print(f"Averaged MFCC array: {X_mfcc_avg.shape}")
print(f"Labels array:        {y.shape}  | Dysarthric: {y.sum()} | Healthy: {(y==0).sum()}")

In [ ]:
# Visualise prosodic feature distributions: dysarthric vs healthy
feat_names = ['Jitter', 'Shimmer', 'HNR', 'F0 Mean (Hz)',
              'F0 Std (Hz)', 'Speaking Rate', 'Voice Break %']

df_pros = pd.DataFrame(X_prosodic, columns=feat_names)
df_pros['class'] = ['Dysarthric' if yi == 1 else 'Healthy' for yi in y]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(feat_names):
    for cls, color in zip(['Healthy', 'Dysarthric'], ['steelblue', 'coral']):
        subset = df_pros[df_pros['class'] == cls][feat]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=cls, edgecolor='white')
    axes[i].set_title(feat)
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel('Value')

axes[-1].axis('off')
plt.suptitle('Prosodic Feature Distributions: Healthy vs Dysarthric', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('prosodic_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Splitting and Normalisation

Stratified 75/25 train-test split. Normalisation is fit on training data only to prevent data leakage.

In [ ]:
# Stratified split — 75% train, 25% test
indices = np.arange(len(y))
(idx_train, idx_test) = train_test_split(
    indices, test_size=0.25, stratify=y, random_state=42
)

# ── MFCC sequences (for 1D-CNN) ──
X_seq_train = X_mfcc_seq[idx_train]
X_seq_test  = X_mfcc_seq[idx_test]

# Normalise per-feature across time (fit on train only)
mean_seq = X_seq_train.mean(axis=(0, 1), keepdims=True)
std_seq  = X_seq_train.std(axis=(0, 1), keepdims=True) + 1e-8
X_seq_train = (X_seq_train - mean_seq) / std_seq
X_seq_test  = (X_seq_test  - mean_seq) / std_seq

# ── Prosodic features ──
X_pro_train = X_prosodic[idx_train]
X_pro_test  = X_prosodic[idx_test]

pro_scaler = StandardScaler()
X_pro_train = pro_scaler.fit_transform(X_pro_train)
X_pro_test  = pro_scaler.transform(X_pro_test)

# ── Averaged MFCCs (for SVM baseline) ──
X_avg_train = X_mfcc_avg[idx_train]
X_avg_test  = X_mfcc_avg[idx_test]

avg_scaler = StandardScaler()
X_avg_train = avg_scaler.fit_transform(X_avg_train)
X_avg_test  = avg_scaler.transform(X_avg_test)

# Labels
y_train = y[idx_train]
y_test  = y[idx_test]

print(f"Train: {len(idx_train)} samples | Test: {len(idx_test)} samples")
print(f"Train — Dysarthric: {y_train.sum()} | Healthy: {(y_train==0).sum()}")
print(f"Test  — Dysarthric: {y_test.sum()}  | Healthy: {(y_test==0).sum()}")
print(f"\nX_seq_train shape: {X_seq_train.shape}")
print(f"X_pro_train shape: {X_pro_train.shape}")

## 5. Model Definitions

### Baseline A — SVM on Frame-Averaged MFCCs (replicates prior work)
### Model B — 1D-CNN on Temporal MFCC Sequences
### Model C (Proposed) — 1D-CNN + Prosodic Late Fusion

In [ ]:
def build_model_b(input_shape=(MAX_FRAMES, N_MFCC)):
    """
    Model B: 1D-CNN on temporal MFCC sequence.
    Input: (batch, T, n_mfcc)
    """
    inp = Input(shape=input_shape, name='mfcc_sequence')

    x = Conv1D(64, kernel_size=3, padding='same', name='conv1')(inp)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(128, kernel_size=3, padding='same', name='conv2')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(128, kernel_size=3, padding='same', name='conv3')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # Global average pooling — learns temporal summary without losing dynamics
    x = GlobalAveragePooling1D(name='temporal_pool')(x)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    out = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs=inp, outputs=out, name='Model_B_1DCNN')
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


# Preview architecture
model_b_preview = build_model_b()
model_b_preview.summary()

In [ ]:
def build_model_c(mfcc_shape=(MAX_FRAMES, N_MFCC), prosodic_dim=N_PROSODIC):
    """
    Model C (Proposed): Late fusion of 1D-CNN temporal MFCC branch
    and prosodic feature branch.

    Architecture:
      Branch 1: MFCC sequence → Conv1D blocks → GlobalAveragePool → 128-dim embedding
      Branch 2: Prosodic vector → Dense(32) → 32-dim embedding
      Fusion:   Concatenate → Dense(64) → Dropout → Sigmoid

    The fusion happens AFTER the CNN has learned temporal representations,
    allowing each modality to specialise independently before combination.
    """
    # ── Branch 1: Temporal MFCC CNN ──────────────────────────
    mfcc_input = Input(shape=mfcc_shape, name='mfcc_input')

    x = Conv1D(64, kernel_size=3, padding='same', name='cnn_conv1')(mfcc_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(128, kernel_size=3, padding='same', name='cnn_conv2')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(128, kernel_size=3, padding='same', name='cnn_conv3')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    mfcc_embedding = GlobalAveragePooling1D(name='mfcc_embedding')(x)

    # ── Branch 2: Prosodic Feature MLP ───────────────────────
    prosodic_input = Input(shape=(prosodic_dim,), name='prosodic_input')
    p = Dense(32, activation='relu', name='pros_dense1')(prosodic_input)
    p = Dropout(0.3)(p)
    prosodic_embedding = Dense(32, activation='relu', name='pros_dense2')(p)

    # ── Late Fusion ───────────────────────────────────────────
    fused = Concatenate(name='late_fusion')([mfcc_embedding, prosodic_embedding])
    fused = Dense(64, activation='relu', name='fusion_dense')(fused)
    fused = Dropout(0.4)(fused)
    output = Dense(1, activation='sigmoid', name='output')(fused)

    model = Model(
        inputs=[mfcc_input, prosodic_input],
        outputs=output,
        name='Model_C_LateFusion'
    )
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


model_c_preview = build_model_c()
model_c_preview.summary()

## 6. Evaluation Utilities

In [ ]:
def compute_metrics(y_true, y_pred_prob, threshold=0.5, model_name='Model'):
    """Compute and print full evaluation metrics for a binary classifier."""
    y_pred = (y_pred_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy    = accuracy_score(y_true, y_pred)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # Recall for dysarthric class
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # Recall for healthy class
    f1          = f1_score(y_true, y_pred)
    auc         = roc_auc_score(y_true, y_pred_prob)

    metrics = {
        'Model':       model_name,
        'Accuracy':    round(accuracy * 100, 2),
        'Sensitivity': round(sensitivity * 100, 2),
        'Specificity': round(specificity * 100, 2),
        'F1-Score':    round(f1, 4),
        'AUC-ROC':     round(auc, 4),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    }

    print(f"\n{'='*55}")
    print(f"  Results — {model_name}")
    print(f"{'='*55}")
    print(f"  Accuracy:    {accuracy*100:.2f}%")
    print(f"  Sensitivity: {sensitivity*100:.2f}%  (clinical: true positive rate)")
    print(f"  Specificity: {specificity*100:.2f}%  (clinical: true negative rate)")
    print(f"  F1-Score:    {f1:.4f}")
    print(f"  AUC-ROC:     {auc:.4f}")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(classification_report(y_true, y_pred,
                                target_names=['Healthy', 'Dysarthric']))
    return metrics


def plot_confusion_matrix(y_true, y_pred_prob, title, threshold=0.5, ax=None):
    """Plot a labelled confusion matrix."""
    y_pred = (y_pred_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Healthy', 'Dysarthric'],
        yticklabels=['Healthy', 'Dysarthric'],
        ax=ax, cbar=False, linewidths=0.5
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)


def plot_roc_curves(roc_data, save_path='roc_curves.png'):
    """Overlay ROC curves for multiple models."""
    plt.figure(figsize=(7, 6))
    colors = ['gray', 'steelblue', 'coral']
    for (name, y_true, y_prob), color in zip(roc_data, colors):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC = {auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — Model Comparison')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_training_history(history, title, save_path):
    """Plot training and validation loss/accuracy curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'],     label='Train Loss', color='steelblue')
    axes[0].plot(history.history['val_loss'], label='Val Loss',   color='coral')
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train Acc', color='steelblue')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc',   color='coral')
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print("Evaluation utilities defined.")

## 7. Baseline A — SVM on Frame-Averaged MFCCs

This replicates the approach of the original paper and most prior TORGO work.

In [ ]:
# Train SVM with RBF kernel (standard in literature)
svm = SVC(kernel='rbf', C=1.0, gamma=10.0,
          probability=True, random_state=42)
svm.fit(X_avg_train, y_train)

# Predict
y_prob_svm = svm.predict_proba(X_avg_test)[:, 1]
metrics_svm = compute_metrics(y_test, y_prob_svm, model_name='Baseline A: SVM + Avg MFCC')

## 8. Model B — 1D-CNN on Temporal MFCC Sequences

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 32

callbacks_b = [
    EarlyStopping(monitor='val_loss', patience=7,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('model_b_best.h5', monitor='val_loss',
                    save_best_only=True, verbose=0)
]

model_b = build_model_b()

history_b = model_b.fit(
    X_seq_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_seq_test, y_test),
    callbacks=callbacks_b,
    verbose=1
)

plot_training_history(history_b, 'Model B — 1D-CNN', 'history_model_b.png')

y_prob_b = model_b.predict(X_seq_test, verbose=0).flatten()
metrics_b = compute_metrics(y_test, y_prob_b, model_name='Model B: 1D-CNN + Temporal MFCC')

## 9. Model C (Proposed) — 1D-CNN + Prosodic Late Fusion

In [ ]:
callbacks_c = [
    EarlyStopping(monitor='val_loss', patience=7,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('model_c_best.h5', monitor='val_loss',
                    save_best_only=True, verbose=0)
]

model_c = build_model_c()

history_c = model_c.fit(
    [X_seq_train, X_pro_train], y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=([X_seq_test, X_pro_test], y_test),
    callbacks=callbacks_c,
    verbose=1
)

plot_training_history(history_c, 'Model C — Late Fusion (Proposed)', 'history_model_c.png')

y_prob_c = model_c.predict([X_seq_test, X_pro_test], verbose=0).flatten()
metrics_c = compute_metrics(y_test, y_prob_c, model_name='Model C (Proposed): 1D-CNN + Prosodic Fusion')

## 10. Ablation Study — Results Table

This table is the core contribution table for the paper.

In [ ]:
results_df = pd.DataFrame([
    {
        'Model':        'Baseline A: SVM + Avg MFCC',
        'Features':     'Frame-averaged MFCC (40-dim)',
        'Temporal':     'No',
        'Prosodic':     'No',
        'Accuracy (%)': metrics_svm['Accuracy'],
        'Sensitivity (%)': metrics_svm['Sensitivity'],
        'Specificity (%)': metrics_svm['Specificity'],
        'F1-Score':     metrics_svm['F1-Score'],
        'AUC-ROC':      metrics_svm['AUC-ROC'],
    },
    {
        'Model':        'Model B: 1D-CNN + Temporal MFCC',
        'Features':     'Temporal MFCC sequence (128×40)',
        'Temporal':     'Yes',
        'Prosodic':     'No',
        'Accuracy (%)': metrics_b['Accuracy'],
        'Sensitivity (%)': metrics_b['Sensitivity'],
        'Specificity (%)': metrics_b['Specificity'],
        'F1-Score':     metrics_b['F1-Score'],
        'AUC-ROC':      metrics_b['AUC-ROC'],
    },
    {
        'Model':        'Model C (Proposed): Late Fusion',
        'Features':     'Temporal MFCC (128×40) + Prosodic (7-dim)',
        'Temporal':     'Yes',
        'Prosodic':     'Yes',
        'Accuracy (%)': metrics_c['Accuracy'],
        'Sensitivity (%)': metrics_c['Sensitivity'],
        'Specificity (%)': metrics_c['Specificity'],
        'F1-Score':     metrics_c['F1-Score'],
        'AUC-ROC':      metrics_c['AUC-ROC'],
    },
])

print("\n" + "="*80)
print("ABLATION STUDY — CLASSIFICATION PERFORMANCE COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

results_df.to_csv('ablation_results.csv', index=False)
print("\nSaved to ablation_results.csv")

In [ ]:
# Visualise ablation results
metrics_to_plot = ['Accuracy (%)', 'Sensitivity (%)', 'Specificity (%)', 'F1-Score', 'AUC-ROC']
model_labels = ['Baseline A\n(SVM + Avg MFCC)',
                'Model B\n(1D-CNN Temporal)',
                'Model C\n(Proposed Fusion)']
colors = ['#7f8c8d', '#2980b9', '#e74c3c']

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, metric in zip(axes, metrics_to_plot):
    vals = results_df[metric].values
    bars = ax.bar(model_labels, vals, color=colors, edgecolor='black', width=0.5)
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, max(vals) * 1.15)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle('Ablation Study — Model Performance Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Confusion Matrices and ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_confusion_matrix(y_test, y_prob_svm,
    title='Baseline A\nSVM + Avg MFCC', ax=axes[0])
plot_confusion_matrix(y_test, y_prob_b,
    title='Model B\n1D-CNN Temporal MFCC', ax=axes[1])
plot_confusion_matrix(y_test, y_prob_c,
    title='Model C (Proposed)\n1D-CNN + Prosodic Fusion', ax=axes[2])

plt.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
roc_data = [
    ('Baseline A: SVM + Avg MFCC',          y_test, y_prob_svm),
    ('Model B: 1D-CNN Temporal MFCC',        y_test, y_prob_b),
    ('Model C (Proposed): Late Fusion',      y_test, y_prob_c),
]
plot_roc_curves(roc_data, save_path='roc_curves.png')

## 12. 10-Fold Stratified Cross-Validation

Cross-validation confirms generalisability and avoids overfitting to a single split. Results reported as mean ± std.

In [ ]:
K_FOLDS = 10
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

cv_results = {'svm': [], 'model_b': [], 'model_c': []}

print(f"Running {K_FOLDS}-fold stratified cross-validation...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_mfcc_seq, y)):
    print(f"Fold {fold+1}/{K_FOLDS}")

    # ── Split ─────────────────────────────────────────────
    Xs_tr, Xs_val = X_mfcc_seq[train_idx], X_mfcc_seq[val_idx]
    Xp_tr, Xp_val = X_prosodic[train_idx],  X_prosodic[val_idx]
    Xa_tr, Xa_val = X_mfcc_avg[train_idx],  X_mfcc_avg[val_idx]
    y_tr,  y_val  = y[train_idx],            y[val_idx]

    # ── Normalise (fit on train fold only) ────────────────
    m = Xs_tr.mean(axis=(0,1), keepdims=True)
    s = Xs_tr.std(axis=(0,1),  keepdims=True) + 1e-8
    Xs_tr  = (Xs_tr  - m) / s
    Xs_val = (Xs_val - m) / s

    sc_p = StandardScaler().fit(Xp_tr)
    Xp_tr  = sc_p.transform(Xp_tr)
    Xp_val = sc_p.transform(Xp_val)

    sc_a = StandardScaler().fit(Xa_tr)
    Xa_tr  = sc_a.transform(Xa_tr)
    Xa_val = sc_a.transform(Xa_val)

    # ── Baseline A: SVM ───────────────────────────────────
    svm_cv = SVC(kernel='rbf', C=1.0, gamma=10.0,
                 probability=True, random_state=42)
    svm_cv.fit(Xa_tr, y_tr)
    prob = svm_cv.predict_proba(Xa_val)[:, 1]
    cv_results['svm'].append(roc_auc_score(y_val, prob))

    # ── Model B: 1D-CNN ───────────────────────────────────
    cb = [EarlyStopping(monitor='val_loss', patience=5,
                        restore_best_weights=True, verbose=0)]
    mb = build_model_b()
    mb.fit(Xs_tr, y_tr, epochs=30, batch_size=32,
           validation_data=(Xs_val, y_val),
           callbacks=cb, verbose=0)
    prob = mb.predict(Xs_val, verbose=0).flatten()
    cv_results['model_b'].append(roc_auc_score(y_val, prob))

    # ── Model C: Late Fusion ──────────────────────────────
    mc = build_model_c()
    mc.fit([Xs_tr, Xp_tr], y_tr, epochs=30, batch_size=32,
           validation_data=([Xs_val, Xp_val], y_val),
           callbacks=cb, verbose=0)
    prob = mc.predict([Xs_val, Xp_val], verbose=0).flatten()
    cv_results['model_c'].append(roc_auc_score(y_val, prob))

    # Free memory
    del mb, mc
    tf.keras.backend.clear_session()

print("\n" + "="*55)
print(f"  {K_FOLDS}-FOLD CROSS-VALIDATION RESULTS (AUC-ROC)")
print("="*55)
for key, name in zip(
    ['svm', 'model_b', 'model_c'],
    ['Baseline A: SVM + Avg MFCC',
     'Model B: 1D-CNN Temporal',
     'Model C (Proposed): Fusion']
):
    scores = cv_results[key]
    print(f"  {name}")
    print(f"    Mean AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
    print(f"    Per-fold: {[round(s,4) for s in scores]}\n")

In [ ]:
# Cross-validation results plot
fig, ax = plt.subplots(figsize=(10, 6))

labels_cv = ['Baseline A\n(SVM + Avg MFCC)',
             'Model B\n(1D-CNN Temporal)',
             'Model C\n(Proposed Fusion)']
data_cv   = [cv_results['svm'], cv_results['model_b'], cv_results['model_c']]
colors_cv = ['#7f8c8d', '#2980b9', '#e74c3c']

bp = ax.boxplot(data_cv, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], colors_cv):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticklabels(labels_cv)
ax.set_ylabel('AUC-ROC')
ax.set_title(f'{K_FOLDS}-Fold Cross-Validation AUC-ROC Distribution')
ax.set_ylim(0.5, 1.05)

# Annotate means
for i, d in enumerate(data_cv, 1):
    ax.text(i, np.mean(d) + 0.01, f'μ={np.mean(d):.3f}',
            ha='center', fontsize=10, color='black')

plt.tight_layout()
plt.savefig('cv_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Statistical Significance — McNemar's Test

McNemar's test determines whether the performance difference between models is statistically significant, not just due to random variation. Required for IEEE publication claims.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(y_true, y_pred_a, y_pred_b, name_a, name_b):
    """McNemar's test between two binary classifiers."""
    # Disagreement table
    b = np.sum((y_pred_a == y_true) & (y_pred_b != y_true))  # A correct, B wrong
    c = np.sum((y_pred_a != y_true) & (y_pred_b == y_true))  # A wrong, B correct

    table = np.array([[0, b], [c, 0]])  # McNemar contingency table
    result = mcnemar(table, exact=True)

    sig = '** SIGNIFICANT **' if result.pvalue < 0.05 else 'not significant'
    print(f"  McNemar's Test: {name_a} vs {name_b}")
    print(f"    b={b}, c={c}")
    print(f"    p-value = {result.pvalue:.4f}  →  {sig}")
    print()
    return result.pvalue

# Binary predictions
pred_svm = (y_prob_svm >= 0.5).astype(int)
pred_b   = (y_prob_b   >= 0.5).astype(int)
pred_c   = (y_prob_c   >= 0.5).astype(int)

print("Statistical Significance Tests (α = 0.05)\n")
mcnemar_test(y_test, pred_svm, pred_b, 'Baseline A (SVM)', 'Model B (1D-CNN)')
mcnemar_test(y_test, pred_svm, pred_c, 'Baseline A (SVM)', 'Model C (Fusion)')
mcnemar_test(y_test, pred_b,   pred_c, 'Model B (1D-CNN)', 'Model C (Fusion)')

## 14. Feature Importance Analysis

Permutation-based feature importance for prosodic features — maps contribution back to clinical indicators (Choi et al., 2024).

In [ ]:
prosodic_names = ['Jitter', 'Shimmer', 'HNR',
                  'F0 Mean', 'F0 Std',
                  'Speaking Rate', 'Voice Break %']

# Baseline AUC with all prosodic features intact
base_prob = model_c.predict([X_seq_test, X_pro_test], verbose=0).flatten()
base_auc  = roc_auc_score(y_test, base_prob)

importance_scores = []

for i, name in enumerate(prosodic_names):
    # Permute feature i
    X_pro_permuted = X_pro_test.copy()
    np.random.shuffle(X_pro_permuted[:, i])

    perm_prob = model_c.predict([X_seq_test, X_pro_permuted], verbose=0).flatten()
    perm_auc  = roc_auc_score(y_test, perm_prob)

    drop = base_auc - perm_auc  # AUC drop when feature is permuted
    importance_scores.append((name, drop))

importance_scores.sort(key=lambda x: x[1], reverse=True)

print("Prosodic Feature Importance (Permutation-based AUC drop)")
print(f"Baseline AUC: {base_auc:.4f}\n")
for name, drop in importance_scores:
    print(f"  {name:<20}: AUC drop = {drop:+.4f}")

# Plot
names, drops = zip(*importance_scores)
colors_imp = ['#e74c3c' if d > 0 else '#2980b9' for d in drops]

plt.figure(figsize=(9, 5))
bars = plt.barh(names, drops, color=colors_imp, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('AUC Drop (positive = important)')
plt.title('Prosodic Feature Importance — Permutation-based')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Final Summary Table

In [ ]:
cv_summary = pd.DataFrame({
    'Model': [
        'Baseline A: SVM + Avg MFCC',
        'Model B: 1D-CNN + Temporal MFCC',
        'Model C (Proposed): Late Fusion'
    ],
    'Accuracy (%)': [
        metrics_svm['Accuracy'],
        metrics_b['Accuracy'],
        metrics_c['Accuracy']
    ],
    'Sensitivity (%)': [
        metrics_svm['Sensitivity'],
        metrics_b['Sensitivity'],
        metrics_c['Sensitivity']
    ],
    'Specificity (%)': [
        metrics_svm['Specificity'],
        metrics_b['Specificity'],
        metrics_c['Specificity']
    ],
    'F1-Score': [
        metrics_svm['F1-Score'],
        metrics_b['F1-Score'],
        metrics_c['F1-Score']
    ],
    'AUC-ROC (test)': [
        metrics_svm['AUC-ROC'],
        metrics_b['AUC-ROC'],
        metrics_c['AUC-ROC']
    ],
    'CV AUC Mean': [
        round(np.mean(cv_results['svm']), 4),
        round(np.mean(cv_results['model_b']), 4),
        round(np.mean(cv_results['model_c']), 4)
    ],
    'CV AUC Std': [
        round(np.std(cv_results['svm']), 4),
        round(np.std(cv_results['model_b']), 4),
        round(np.std(cv_results['model_c']), 4)
    ]
})

print("\n" + "="*90)
print("  FINAL RESULTS SUMMARY — TORGO DATABASE")
print("="*90)
print(cv_summary.to_string(index=False))
print("="*90)
print("\nKey finding: Model C (Late Fusion) demonstrates whether")
print("combining temporal MFCC dynamics with prosodic features")
print("outperforms frame-averaged classical approaches.")

cv_summary.to_csv('final_results.csv', index=False)
print("\nSaved to final_results.csv")

## References

- Rudzicz, F., Namasivayam, A. K., & Wolff, T. (2012). The TORGO database of acoustic and articulatory speech from speakers with dysarthria. *Language Resources and Evaluation*, 46(4), 523–541.
- Hernandez, A., Kim, S., & Chung, M. (2020). Prosody-based measures for automatic severity assessment of dysarthric speech. *Applied Sciences*, 10(19), 6999.
- Joshy, A. A., & Rajan, R. (2022). Automated dysarthria severity classification: A study on acoustic features and deep learning techniques. *IEEE TNSRE*, 30, 1147–1157.
- Al-Ali, A., et al. (2024). The detection of dysarthria severity levels using AI models: A review. *IEEE Access*, 12, 48223–48238.
- Choi, Y., Lee, J., & Koo, M.-W. (2024). Speech recognition-based feature extraction for enhanced automatic severity classification in dysarthric speech. *arXiv:2412.03784*.
- Anuprabha, M., et al. (2024). A multi-modal approach to dysarthria detection and severity assessment. *arXiv:2412.16874*.